# ColdStart Killer — Insert 3K MVP Dataset into MongoDB

⚠️ Run only after:
1. Notebook 01 has generated `analysis/mvp_3000_items_diverse.csv`
2. MongoDB Atlas cluster is ready
3. `.env` is filled

Atlas indexes are not required for MongoDB writes. If Atlas needs existing data before index creation, run the small write first, create `vector_index` and `text_index` on `retrieval_units`, then continue larger batches with `resume=True`. Retrieval tests require both indexes to be READY.

Pipeline input: `analysis/mvp_3000_items_diverse.csv`, where each row is one cold-start product with canonical metadata and `product_text_for_llm`.

Pipeline output:
- `items`: one business product document per row, with catalog metadata and source text.
- `retrieval_units`: multiple semantic entry points per product. `hype_question` units store 1024-dimensional embeddings for Vector Search; `proposition` units store grounded facts for Atlas Search/BM25.

Insert flow diagram:

![Notebook 02 insert flow](../assets/insert_flow.svg)

Short version: CSV row -> `items` document + `proposition` retrieval units for BM25 + `hype_question` retrieval units for Vector Search.

## 1. Setup imports

Load project modules and MongoDB/indexing helpers. This notebook assumes Notebook 01 already produced the MVP CSV in `analysis/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.indexing import (
    build_contextual_header,
    build_hype_units,
    build_item_doc_from_mvp_row,
    build_proposition_units,
    estimate_indexing_size,
    index_items_from_dataframe,
)
from src.llm_hype import generate_hype_queries_llm
from src.llm_propositions import extract_propositions_llm
from src.embeddings import embed_texts
from src.mongodb import collection_counts, ping_mongodb
from src.validation import assert_required_columns

## 2. Load env and connect MongoDB

Load `.env`, connect to MongoDB Atlas, and verify that credentials and network access work before running expensive LLM or embedding steps.

In [2]:
ping_mongodb()

{'ok': True, 'result': {'ok': 1}, 'database': 'coldstart_killer'}

## 3. Load `analysis/mvp_3000_items_diverse.csv`

Read the primary MVP insert CSV created by Notebook 01. This CSV is the batch indexing input: each row contains canonical product metadata plus `product_text_for_llm`, then becomes one `items` document and several `retrieval_units`.

In [3]:
csv_path = ROOT / "analysis" / "mvp_3000_items_diverse.csv"
df = pd.read_csv(csv_path)
df.shape

(3000, 28)

## 4. Validate required columns

Fail fast if the CSV is missing fields required by the schema and indexing pipeline. This prevents partial inserts caused by malformed input data.

In [4]:
assert_required_columns(df)
"required columns OK"

'required columns OK'

## 5. Estimate indexing size

Estimate expected item count, HyPE vectors, proposition units, retrieval units, and raw vector memory before writing to MongoDB. Only HyPE units are embedded; proposition units stay text-only for Atlas Search/BM25.

In [5]:
estimate_indexing_size(df)

{'item_count': 3000,
 'expected_hype_vectors': 13500,
 'expected_propositions': 16500,
 'expected_retrieval_units': 30000,
 'raw_vector_memory': {'num_vectors': 13500,
  'dimensions': 1024,
  'raw_float32_bytes': 55296000,
  'raw_float32_mb': 52.734,
  'list_float_note': 'MongoDB list floats have BSON overhead beyond raw float32 memory.',
  'norm_expected': 1.0,
  'cosine_ready': True}}

## 6. Dry run 3 items

Run the full generation and embedding flow on a few items without MongoDB writes. This checks the full offline indexing path: item row -> propositions -> HyPE queries -> BGE-M3 embeddings -> MongoDB document shapes.

In [6]:
dry_run_3 = index_items_from_dataframe(df, limit=3, dry_run=True, sleep_seconds=0.5)
dry_run_3

c:\HCMUS\MONGODB\coldstart+project\ColdStart_Killer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]


{'dry_run': True,
 'resume': False,
 'start_index': 0,
 'end_index_exclusive': 3,
 'requested_limit': 3,
 'item_count': 3,
 'hype_units': 15,
 'proposition_units': 19,
 'failed_items': [],
 'runtime_seconds': 114.916,
 'estimated_raw_vector_memory_mb': 0.055,
 'estimated_retrieval_units': 30}

## 7. Show sample item doc

Build one `items` document from the CSV so the product metadata shape can be inspected before insertion. The product is the business entity here: it stores catalog metadata, source text, enrichment fields, and cold-start flags, but no retrieval embedding.

In [7]:
sample_item = build_item_doc_from_mvp_row(df.iloc[0])
sample_item

{'_id': 'B0BNK971SB',
 'raw_parent_asin': 'B0BNK971SB',
 'source_dataset': 'Amazon Reviews 2023',
 'source_file': 'analysis/mvp_3000_items_diverse.csv',
 'source_category': 'Cell_Phones_and_Accessories',
 'title_en': 'USB C Samsung Fast Charging Block Plug for Samsung Galaxy A14 5G,A54,A23,A13 5G,A53,A24,S23,A03s,A34,Z Fold4,S21FE,S22,Type C Charger Box Wall Adapter for iPhone 14,13 Pro Max,12,11,8,X;Pixel 7 Pro,6a',
 'title_vi': '',
 'brand': 'GiGreen',
 'brand_source': 'details.Brand',
 'category_id': 'all_electronics',
 'category_path': ['Cell Phones & Accessories',
  'Accessories',
  'Chargers & Power Adapters',
  'Wall Chargers'],
 'raw_main_category': 'All Electronics',
 'price_usd': 8.49,
 'price_vnd': 212250,
 'price_parse_status': 'parsed',
 'price_bucket': '100k_300k',
 'in_stock': True,
 'image_url': 'https://m.media-amazon.com/images/I/31d84uNbTYL._AC_SR38,50_.jpg',
 'image_urls': ['https://m.media-amazon.com/images/I/31d84uNbTYL._AC_SR38,50_.jpg',
  'https://m.media-amazon

## 8. Show sample proposition units

Generate grounded proposition units for one sample item. Proposition units are atomic product facts with `raw_text` and `text_search`; they support Atlas Search/BM25 fact matching and do not store vector embeddings.

In [8]:
sample_propositions = extract_propositions_llm(sample_item)
sample_prop_units = build_proposition_units(sample_item, sample_propositions)
sample_prop_units[:3]

[{'_id': '9cfa2a88-43ae-478f-bcae-76d2887584ec',
  'item_id': 'B0BNK971SB',
  'unit_type': 'proposition',
  'language': 'en',
  'raw_text': 'Output power is 5V 3A, 9V 2A, 12V 1.5A at 20W.',
  'confidence': 0.95,
  'source': 'llm',
  'category_id': 'all_electronics',
  'price_vnd': 212250,
  'price_bucket': '100k_300k',
  'in_stock': True,
  'is_cold_item': True,
  'seller_confirmed': False,
  'generation_model': 'qwen3:8b',
  'generation_prompt_version': 'proposition_v1',
  'proposition_type': 'spec',
  'text_search': 'Output power is 5V 3A, 9V 2A, 12V 1.5A at 20W.',
  'item_title_en': 'USB C Samsung Fast Charging Block Plug for Samsung Galaxy A14 5G,A54,A23,A13 5G,A53,A24,S23,A03s,A34,Z Fold4,S21FE,S22,Type C Charger Box Wall Adapter for iPhone 14,13 Pro Max,12,11,8,X;Pixel 7 Pro,6a',
  'item_brand': 'GiGreen',
  'source_field': 'description'},
 {'_id': '14749e3a-a8f1-407a-a250-f652d1ef7551',
  'item_id': 'B0BNK971SB',
  'unit_type': 'proposition',
  'language': 'en',
  'raw_text': 'I

## 9. Show sample HyPE units

Generate HyPE buyer search queries, embed them, and build vector-search retrieval units. The embedding input is `[Category | Brand | Price bucket] + HyPE query`, producing 1024-dimensional BGE-M3 vectors stored on `retrieval_units.embedding`.

In [9]:
sample_hype = generate_hype_queries_llm(sample_item, sample_propositions)
sample_embedding_texts = [f"{build_contextual_header(sample_item)} {query['raw_text']}" for query in sample_hype]
sample_embeddings = embed_texts(sample_embedding_texts)
sample_hype_units = build_hype_units(sample_item, sample_hype, sample_embeddings)
sample_hype_units[:3]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.82it/s]


[{'_id': '1dff0125-0cdd-444d-a2d3-c128410cc043',
  'item_id': 'B0BNK971SB',
  'unit_type': 'hype_question',
  'language': 'en',
  'raw_text': 'Fast charge samsung galaxy a14 5g in 30 minutes',
  'confidence': 0.95,
  'source': 'llm',
  'category_id': 'all_electronics',
  'price_vnd': 212250,
  'price_bucket': '100k_300k',
  'in_stock': True,
  'is_cold_item': True,
  'seller_confirmed': False,
  'generation_model': 'qwen3:8b',
  'generation_prompt_version': 'hype_v1',
  'aspect': 'function',
  'embedding_text': '[Category: All Electronics | Brand: GiGreen | Price bucket: 100k_300k] Fast charge samsung galaxy a14 5g in 30 minutes',
  'embedding': [-0.018194489181041718,
   -0.0061751171015203,
   -0.030269665643572807,
   -0.012700466439127922,
   0.0018172338604927063,
   -0.024130554869771004,
   -0.00894998386502266,
   -0.004316568374633789,
   0.028002595528960228,
   0.018193956464529037,
   -0.007156949955970049,
   0.03010675497353077,
   -0.005937000270932913,
   0.000776942819

## 10. Insert 10 items

Write a small first batch to verify MongoDB inserts end to end. For each row, the pipeline upserts the `items` document, generates proposition and HyPE units, embeds HyPE only, then replaces that item's retrieval units.

In [ ]:
insert_10 = index_items_from_dataframe(df, limit=10, dry_run=False, sleep_seconds=0.5)
insert_10

## 11. Count MongoDB collections

Check collection counts after writes to confirm that `items` and `retrieval_units` were inserted as expected.

In [10]:
collection_counts()

{'ok': True, 'items': 3000, 'retrieval_units': 29753}

## 12. Insert 100 items

Continue indexing a larger batch after the small write succeeds. Important: without `resume=True` or `start_index`, `limit=100` means process the first 100 CSV rows again; with `resume=True`, the function counts existing MongoDB items and starts from the next missing CSV row.

In [ ]:
insert_100 = index_items_from_dataframe(
    df,
    limit=90,
    dry_run=False,
    sleep_seconds=0.5,
    resume=True,
)
insert_100

## 13. Optional larger inserts

Scale up in controlled batches after indexes are ready and storage usage looks safe. Use `resume=True` for normal continuation, or `start_index` when you intentionally want to begin from a specific CSV row.

In [ ]:
# insert_500 = index_items_from_dataframe(df, limit=500, dry_run=False, sleep_seconds=0.5, resume=True)
# insert_1000 = index_items_from_dataframe(df, limit=1000, dry_run=False, sleep_seconds=0.5, resume=True)
# insert_3000 = index_items_from_dataframe(df, limit=3000, dry_run=False, sleep_seconds=0.5, resume=True)

## 14. Final stats

Summarize final collection counts and estimated retrieval footprint after indexing. Use this as the final sanity check before retrieval testing: `items` should track product rows, while `retrieval_units` should be much larger because each product has multiple semantic entry points.

In [11]:
{
    "counts": collection_counts(),
    "estimated_full_size": estimate_indexing_size(df),
}

{'counts': {'ok': True, 'items': 3000, 'retrieval_units': 29753},
 'estimated_full_size': {'item_count': 3000,
  'expected_hype_vectors': 13500,
  'expected_propositions': 16500,
  'expected_retrieval_units': 30000,
  'raw_vector_memory': {'num_vectors': 13500,
   'dimensions': 1024,
   'raw_float32_bytes': 55296000,
   'raw_float32_mb': 52.734,
   'list_float_note': 'MongoDB list floats have BSON overhead beyond raw float32 memory.',
   'norm_expected': 1.0,
   'cosine_ready': True}}}